## One source of randomness

We have two processes in front of us.

**The stock price** follows geometric Brownian motion:

$$
dS_t = \mu S_t \, dt + \sigma S_t \, dB_t.
$$

**A derivative on the stock**, with price $f(S,t)$, follows Itô's lemma:

$$
df
=
\left(
f_t + \mu S f_S + \frac{1}{2}\sigma^2 S^2 f_{SS}
\right) dt
+
\sigma S f_S \, dB_t.
$$

Look at the last term of each equation.

- The stock's random part is $\sigma S \, dB_t$.
- The derivative's random part is $\sigma S f_S \, dB_t$.

**They share the same $dB_t$.** The same Brownian shock drives both the stock and the derivative. The magnitudes differ by the factor $f_S$, but the source is identical.

This single key observation is the foundation of Black Scholes. If two things are driven by the same random noise, we can combine them in the right proportion to cancel that noise entirely.

## Why does the expected payoff matter?

If you want to price a derivative, the standard formula is

$$
\text{price today} = e^{-rT} \cdot \mathbb{E}[\text{payoff at } T].
$$

For a European call:

$$
C_0 = e^{-rT} \cdot \mathbb{E}[\max(S_T - K, 0)].
$$

To compute the expectation, we need to know the **probability distribution of $S_T$**. And that distribution depends on the drift $\mu$ which is the expected return of the stock.

But here is the problem: **nobody knows $\mu$.**

- You think the stock will return 10%.
- I think it will return 5%.
- The market thinks something else.

If the derivative's price depends on $\mu$, then we cannot agree on the price as we have a huge stock universe and getting hold of a valid $\mu$ would itself be a challenging task. And that means we cannot price these derivatives accurately.

**This is the puzzle Black and Scholes solved.** They found a way to price the derivative *without* needing to know $\mu$  by building a portfolio whose randomness cancels, and whose expected payoff was known.

Before we get there, let's build the intuition with a simple example.

## The casino analogy

Imagine you are in a casino. There is one coin. Every game in the casino is driven by flips of this same coin. Two of the games are:

**Game A:**
- Heads: you win $10.
- Tails: you lose $20.

**Game B:**
- Heads: you win $15.
- Tails: you lose $30.

Both games depend on the same coin. **Neither game, on its own, can be priced without knowing the probability of heads.** If the coin is fair, Game A has expected value

$$
\frac{1}{2}(10) + \frac{1}{2}(-20) = -5.
$$

If heads comes up 60% of the time, the expected value is

$$
0.6(10) + 0.4(-20) = -2.
$$

Different answers. Different prices. Unless we know the probability of heads, we cannot price Game A. Same for Game B.

Now suppose you take a **combination**:

- **Long 3 units of Game A**
- **Short 2 units of Game B**

What is your payoff?

| Outcome | From 3 × Game A | From −2 × Game B | Total |
|---|---|---|---|
| Heads | $3 \times 10 = 30$ | $-2 \times 15 = -30$ | $0$ |
| Tails | $3 \times (-20) = -60$ | $-2 \times (-30) = +60$ | $0$ |

**You get $0 regardless of the coin.** The randomness has cancelled. The combination is **riskless**. More importantly we know the expected payoff for this portfolio.

And because it is riskless, its value today must be exactly $0$. If it cost more than $0$, you could short it for free money. If it cost less than $0$, you could buy it for free money.

**Notice what just happened.** We priced a combination of the two games **without knowing the probability of heads.** The randomness cancelled, and no arbitrage conditionality did the rest.

This is exactly the idea behind Black–Scholes.

In [ ]:
import numpy as np
import plotly.graph_objects as go

A_heads, A_tails = 10, -20
B_heads, B_tails = 15, -30
w_A, w_B = 3, -2
port_heads = w_A * A_heads + w_B * B_heads
port_tails = w_A * A_tails + w_B * B_tails

p_values = np.linspace(0, 1, 41)

EV_A    = p_values * A_heads    + (1 - p_values) * A_tails
EV_B    = p_values * B_heads    + (1 - p_values) * B_tails
EV_port = p_values * port_heads + (1 - p_values) * port_tails

# --- Build frames ---
def bar_frame(i):
    return go.Bar(
        x=["Game A", "Game B", "Portfolio"],
        y=[EV_A[i], EV_B[i], EV_port[i]],
        marker_color=["#dc2626", "#2563eb", "#16a34a"],
        text=[f"${EV_A[i]:.2f}", f"${EV_B[i]:.2f}", f"${EV_port[i]:.2f}"],
        textposition="outside",
        showlegend=False,
    )

frames = [
    go.Frame(data=[bar_frame(i)], name=f"{p:.2f}")
    for i, p in enumerate(p_values)
]

fig = go.Figure(
    data=[bar_frame(0)],
    frames=frames,
    layout=go.Layout(
        title="Expected value as the coin bias changes",
        yaxis=dict(title="Expected value ($)", range=[-35, 35]),
        xaxis=dict(title=""),
        height=520,
        updatemenus=[dict(
            type="buttons",
            showactive=False,
            x=0.05, y=1.15, xanchor="left",
            buttons=[dict(
                label="▶ Play",
                method="animate",
                args=[None, {"frame": {"duration": 250, "redraw": True},
                             "fromcurrent": True,
                             "transition": {"duration": 80}}],
            )],
        )],
        sliders=[dict(
            active=20,
            currentvalue=dict(prefix="P(heads) = ", font=dict(size=14)),
            pad=dict(t=40),
            steps=[
                dict(
                    method="animate",
                    args=[[f"{p:.2f}"],
                          {"frame": {"duration": 0, "redraw": True},
                           "mode": "immediate"}],
                    label=f"{p:.2f}",
                )
                for p in p_values
            ],
        )],
    ),
)

fig.show()

### A small digression on the casino

You might object, why would the casino ever change its coin? Well,
maybe they let you in with $P(\text{heads}) > 0.5$ to get you hooked,
and once you're deep in the game they quietly slip in a weighted coin
with $P(\text{heads}) < 0.5$. A little unethical, but it
makes the point.

The important observation is: **we never needed to know $P(\text{heads})$
to price the portfolio.** Whatever the casino does to the coin, the
portfolio pays the same amount. The coin bias is irrelevant.


### Mapping back to our problem

Now translate the casino back to finance.

| Casino | Finance |
|---|---|
| The coin flip | The Brownian shock $dB_t$ |
| The coin's bias $P(\text{heads})$ | The drift $\mu$ (expected return) |
| Game A | The stock $S_t$ |
| Game B | The derivative $f(S_t, t)$ |
| The ratio $3 : -2$ | The hedge ratio $f_S : -1$ |

Look at the second row. The coin's bias plays the role of $\mu$. We
never needed it. We never learned it. We never will. And yet the price
of the riskless portfolio was pinned down completely by no arb
alone.

### Coming back to the stochastic side

In the casino, the source of randomness was the coin flip. In our
problem, the source of randomness is the Brownian increment $dB_t$.

We have already seen that both the stock price and the derivative
price are driven by the **same** $dB_t$:

$$
dS = \mu S \, dt + \sigma S \, dB_t,
$$

$$
df = \left(f_t + \mu S f_S + \tfrac{1}{2}\sigma^2 S^2 f_{SS}\right) dt + \sigma S f_S \, dB_t.
$$

The random term in $dS$ is $\sigma S \, dB_t$. The random term in $df$
is $\sigma S f_S \, dB_t$. They differ only by the constant factor
$f_S$. That factor is exactly the "ratio" we need.

Hold $f_S$ shares of stock against one unit of the derivative, and the
$dB_t$ terms cancel:

$$
f_S \cdot \sigma S \, dB_t - \sigma S f_S \, dB_t = 0.
$$

The portfolio is now **riskless**. Its value does not depend on which
way the Brownian shock moves — just as the casino portfolio's value did
not depend on which way the coin landed.

That is the key idea. The rest of the Black–Scholes derivation is
just bookkeeping: set the return of this riskless portfolio equal to
$r$, and read off the PDE.

## The no-arbitrage condition

We have a portfolio $\Pi = -f + f_S S$ whose randomness cancels.
Its change $d\Pi$ is therefore deterministic. Call its rate of return
$\rho$, so that

$$
d\Pi = \rho\, \Pi \, dt.
$$

What must $\rho$ be? The answer is forced by no-arbitrage.

**If $\rho > r$:** borrow $\Pi$ at rate $r$, buy the portfolio, earn
$(\rho - r)\Pi\,dt$ for free. Arbitrage.

**If $\rho < r$:** short the portfolio, deposit $\Pi$ at rate $r$,
earn $(r - \rho)\Pi\,dt$ for free. Arbitrage.

Both are impossible in a well functioning market. So the only
consistent value is

$$
\boxed{\rho = r.}
$$

A riskless portfolio must earn **exactly** the risk free rate. Not
more otherwise everyone borrows to buy it. Not less otherwise
everyone shorts it and deposits the proceeds.

This single condition is what gives us the Black Scholes PDE.